# 02 - Baseline Modeller ve Model Kıyaslama Analizi (Ö2)

Bu notebook, **Zaman Serisi ve Değerlendirme Lideri (Ö2)** tarafından projenin baseline modelleri ile makine öğrenmesi modellerinin karşılaştırmalı performansını incelemek ve proje başarı kriteri eşiğini hesaplamak için hazırlanmıştır.

### Değerlendirme Çerçevesi:
- **Doğrulama Metodu:** 3 Katlı Genişleyen Pencere (*Expanding Window Backtest*)
- **Tahmin Ufku:** $H = 28$ Gün
- **Veri Sızıntısı Koruması:** $Lag \ge 28$ gün
- **Başarı Kriteri:** ML modelinin, en iyi baseline modelden en az **%10 daha düşük MAE** üretmesi.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# 1. Backtest sonuç tablosunu yükle
results_file = Path('../results/model_comparison.csv')
if not results_file.exists():
    results_file = Path('results/model_comparison.csv')

df = pd.read_csv(results_file)
print('=== TÜM MODELLERİN BACKTEST METRİKLERİ ===')
print(df.to_string(index=False))

## 2. En İyi Baseline Modelin Tespiti ve Başarı Eşiği Hesabı

In [ ]:
baselines = ['persistence', 'seasonal_naive_7', 'ma_7', 'ma_28']
df_base = df[df['model'].isin(baselines)].copy()
best_base = df_base.loc[df_base['MAE'].idxmin()]
best_mae = float(best_base['MAE'])

target_threshold = best_mae * 0.90
print(f"En İyi Baseline Model : {best_base['model'].upper()}")
print(f"Baseline MAE          : {best_mae:.4f} adet/gün")
print(f"Hedef MAE (-%10)      : {target_threshold:.4f} adet/gün\n")

# ML modellerinin hedefi aşma durumu
df_ml = df[~df['model'].isin(baselines)].copy()
df_ml['improvement_%'] = ((best_mae - df_ml['MAE']) / best_mae) * 100
df_ml['target_met'] = df_ml['MAE'] <= target_threshold
print(df_ml[['model', 'MAE', 'improvement_%', 'target_met']].to_string(index=False))

## 3. Metrik Karşılaştırma Grafiği

In [ ]:
plt.figure(figsize=(9, 4.5))
df_sorted = df.sort_values('MAE')
colors = ['#27ae60' if m == 'lightgbm' else '#2980b9' if m in ['xgboost', 'random_forest'] else '#7f8c8d' for m in df_sorted['model']]

bars = plt.barh(df_sorted['model'], df_sorted['MAE'], color=colors)
plt.axvline(best_mae, color='#c0392b', linestyle='--', linewidth=1.5, label=f'En İyi Baseline (MA-28: {best_mae:.2f})')
plt.axvline(target_threshold, color='#27ae60', linestyle=':', linewidth=2, label=f'Hedef Eşik (-%10: {target_threshold:.2f})')
plt.title('Baseline vs ML Modelleri MAE Karşılaştırması (Ufuk = 28 Gün)', fontsize=12, fontweight='bold')
plt.xlabel('Ortalama Mutlak Hata (MAE - Adet/Gün)')
plt.legend(loc='lower right')
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()